In [1]:
import anndata as ad
import scipy.sparse as sp
import os
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance

from infer_wscreni import GenePeakOverlapLabs, infer_wScReNI_sc_networks

In [2]:
DATASET = "retinal"
BASE    = f"../../../data/processed/{DATASET}"

rna      = ad.read_h5ad(f"{BASE}_rna_sub.h5ad")
knn      = np.load(f"{BASE}_knn_indices.npy")
triplets = pd.read_csv(f"{BASE}_triplets.csv")
labels   = pd.read_csv(f"{BASE}_gene_labels.csv")

peak_matrix   = np.load(f"../../../data/processed/peak_matrix.npy")


In [3]:
X = rna.X.toarray() if sp.issparse(rna.X) else rna.X
expr_df = pd.DataFrame(
        X.T,
        index   = rna.var_names,
        columns = rna.obs['_original_rna_cell'],
    )

peak_names = triplets["peak"].unique()
peak_df = pd.DataFrame(
        peak_matrix.T,
        index   = peak_names,
        columns = rna.obs_names,
    )

In [4]:
rows = []
for _, row in labels.iterrows():
        gene  = row["gene"]
        gtype = row["type"]

        if pd.isna(row["associated_peaks"]):
            continue
        else:
            peaks_for_gene = str(row["associated_peaks"]).split(",")
            TF_str = (
                ";".join(str(row["associated_TFs"]).split(","))
                if pd.notna(row["associated_TFs"]) else ""
            )
            for peak in peaks_for_gene:
                rows.append({"gene": gene, "peak": peak.strip(), "TF": TF_str, "label": gtype})

labs_df = pd.DataFrame(rows)

In [5]:
labs = GenePeakOverlapLabs(
        genes  = labs_df["gene"].tolist(),
        peaks  = labs_df["peak"].tolist(),
        TFs    = labs_df["TF"].tolist(),
        labels = labs_df["label"].tolist(),
    )

print(f"expr_matrix : {expr_df.shape}  (genes × cells)")
print(f"peak_df     : {peak_df.shape}  (peaks × cells)")
print(f"knn         : {knn.shape}  (cells × neighbors)")
print(f"labs entries: {len(labs.genes)} gene-peak rows")
print(f"TF genes    : {(labels['type'] == 'TF').sum()}, "
        f"target genes: {(labels['type'] == 'target').sum()}")

expr_matrix : (500, 400)  (genes × cells)
peak_df     : (217, 400)  (peaks × cells)
knn         : (400, 20)  (cells × neighbors)
labs entries: 228 gene-peak rows
TF genes    : 44, target genes: 456


In [6]:
networks, oob_scores = infer_wScReNI_sc_networks(
        expr_matrix              = expr_df,
        gene_peak_overlap_matrix = peak_df,
        gene_peak_overlap_labs   = labs,
        nearest_neighbors_idx    = knn,
        cell_index               = None,
        nthread                  = 8,
        max_cell_per_batch       = 10,
        seed= 42,
    )

Total number of cells: 400
Cell 0 to cell 9
Cell 10 to cell 19
Cell 20 to cell 29
Cell 30 to cell 39
Cell 40 to cell 49
Cell 50 to cell 59
Cell 60 to cell 69
Cell 70 to cell 79
Cell 80 to cell 89
Cell 90 to cell 99
Cell 100 to cell 109
Cell 110 to cell 119
Cell 120 to cell 129
Cell 130 to cell 139
Cell 140 to cell 149
Cell 150 to cell 159
Cell 160 to cell 169
Cell 170 to cell 179
Cell 180 to cell 189
Cell 190 to cell 199
Cell 200 to cell 209
Cell 210 to cell 219
Cell 220 to cell 229
Cell 230 to cell 239
Cell 240 to cell 249
Cell 250 to cell 259
Cell 260 to cell 269
Cell 270 to cell 279
Cell 280 to cell 289
Cell 290 to cell 299
Cell 300 to cell 309
Cell 310 to cell 319
Cell 320 to cell 329
Cell 330 to cell 339
Cell 340 to cell 349
Cell 350 to cell 359
Cell 360 to cell 369
Cell 370 to cell 379
Cell 380 to cell 389
Cell 390 to cell 399


In [7]:
oob_array = np.stack(oob_scores)


rna.obsm["wScReNI_oob_r2"] = pd.DataFrame(
    oob_array,
    index   = rna.obs_names,
    columns = rna.var_names,
)

rna.write_h5ad(f"output/{DATASET}_with_networks_seed42.h5ad")

np.save(f"output/{DATASET}_networks_seed42.npy", np.stack(networks))